# Medical Necessity — Order Review

Reviews non-emergent ground ambulance transport orders from Transport.net at order time and
produces, for each order, the clinical justification text, a medical necessity determination,
and the reasons behind it.

Knowledge comes from one external reference file, `med_nec_knowledge.json`, which carries
three sources:

- CMS regulatory guidance (Benefit Policy Manual Chapter 10; 42 CFR 410.40, 414.605, 414.640)
- Medical Necessity Facility Training delivered 2026-08-13
- Dave Lyons summary of the discussion with Jen Jones, not yet ingested

The facility training adds two things the earlier scoring model did not have. Bed confinement
is a three-part conjunction that is not on its own sufficient. And there is a set of named
non-covered situations that can rule an order out rather than only failing to rule it in.

## 1. Configuration

In [ ]:
import os
import re
import json
import shutil
from datetime import datetime

import pandas as pd
from pyspark.sql import functions as SF

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

KNOWLEDGE_PATH = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/med_nec_knowledge.json"
SOURCE_TABLE = "`prod-sandbox`.vivekkumar_patel.temp_tnet_tripmaster"
OUTPUT_DIR = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/data"
LOCAL_DIR = "/tmp"

WORKSPACE_BASE_URL = "https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints"
LLM_MODEL = "databricks-gpt-oss-120b"
LLM_TEMPERATURE = 0.1
LLM_MAX_TOKENS = 2500

ORDER_LIMIT = 500
SAMPLE_SEED = 42

EXCLUDED_SERVICE_CODES = ["EMG", "WC", "FWQUOTE", "ORGAN"]

SCORE_NECESSARY_THRESHOLD = 3.0
MIN_TEXT_CHARS = 20

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 200)

## 2. Knowledge reference file

Everything the rules and the model use comes from this file. Editing the JSON changes both
paths at once; no logic below hard-codes a term, a weight, or an exclusion.

In [ ]:
def load_knowledge(path=KNOWLEDGE_PATH):
    with open(path, "r") as f:
        return json.load(f)


KB = load_knowledge()

CONCEPTS = {c["concept"]: c for c in KB["concepts"]}
EXCLUSIONS = {e["exclusion_id"]: e for e in KB["exclusions"]}
BED = KB["bed_confinement"]
BED_ELEMENTS = {e["element"]: e for e in BED["elements"]}

source_frame = pd.DataFrame(KB["sources"])
concept_frame = pd.DataFrame([{
    "concept": c["concept"], "axis": c["axis"], "weight": c["weight"], "status": c["status"],
    "source_id": c["source_id"], "cite": c["cite"], "n_terms": len(c["terms"])
} for c in KB["concepts"]]).sort_values(["axis", "status", "concept"]).reset_index(drop=True)
exclusion_frame = pd.DataFrame([{
    "exclusion_id": e["exclusion_id"], "sole_reason_only": e["sole_reason_only"],
    "source_id": e["source_id"], "cite": e["cite"], "statement": e["statement"]
} for e in KB["exclusions"]])
rule_frame = pd.DataFrame(KB["policy_rules"])

print("knowledge version:", KB["knowledge_version"])
print("governing question:", KB["determination_question"])
print()
print(source_frame.to_string(index=False))
print()
print(concept_frame.to_string(index=False))
print()
print("Bed confinement requires all of:")
for e in BED["elements"]:
    print("  -", e["statement"])
print("  qualifier:", BED["qualifier"])
print()
print(exclusion_frame[["exclusion_id", "sole_reason_only", "cite"]].to_string(index=False))
print()
pending = [s["source_id"] for s in KB["sources"] if s["pending_ingest"]]
print("PENDING_INGEST:", pending if pending else "none")

## 3. Term matchers

One compiled matcher per concept, per bed confinement element, and per exclusion. Word
boundaries are enforced so short tokens do not match inside longer words.

In [ ]:
def compile_matcher(terms):
    parts = [re.escape(t) for t in sorted(set(terms), key=len, reverse=True)]
    return re.compile(r"(?<![a-z0-9])(" + "|".join(parts) + r")(?![a-z0-9])")


CONCEPT_MATCHERS = {k: compile_matcher(v["terms"]) for k, v in CONCEPTS.items()}
BED_MATCHERS = {k: compile_matcher(v["terms"]) for k, v in BED_ELEMENTS.items()}
EXCLUSION_MATCHERS = {k: compile_matcher(v["terms"]) for k, v in EXCLUSIONS.items()}
VAGUE_MATCHER = compile_matcher(KB["non_specific_phrases"])

NEGATION = re.compile(r"\b(no|not|denies|denied|negative for|free of|resolved|able to)\b")
SELF_NEGATING = re.compile(r"^(unable|cannot|can not|non-|non |no )")


def negated(text, span, term, window=30):
    if SELF_NEGATING.search(term):
        return False
    start = max(0, span[0] - window)
    return bool(NEGATION.search(text[start:span[0]]))


def find_terms(text, matchers):
    hits = {}
    for name, rx in matchers.items():
        m = rx.search(text)
        if m and not negated(text, m.span(), m.group(0)):
            hits[name] = m.group(0)
    return hits


print("concept matchers:", len(CONCEPT_MATCHERS),
      "| bed element matchers:", len(BED_MATCHERS),
      "| exclusion matchers:", len(EXCLUSION_MATCHERS))

## 4. Rule-based scorer

`total_score = mobility_score + monitoring_score + named_score`. Bed confinement contributes
to the mobility subtotal only when all three elements are present, which is what the training
requires. A partial claim is recorded and scored at a reduced value so it is visible rather
than silently dropped.

An exclusion marked `sole_reason_only` fires only when nothing else qualifying is documented.
`can_sit_safely_in_wheelchair` and `self_supporting` are not sole-reason exclusions because
they contradict the mobility element directly.

In [ ]:
def score_order(text):
    raw = (text or "").strip()
    low = raw.lower()
    out = {
        "has_text": int(len(raw) >= MIN_TEXT_CHARS),
        "text_len": len(raw),
        "mobility_score": 0.0,
        "monitoring_score": 0.0,
        "named_score": 0.0,
        "total_score": 0.0,
        "bed_elements": [],
        "bed_confined": 0,
        "bed_partial": 0,
        "concept_hits": {},
        "exclusion_hits": {},
        "vague_only": 0,
    }
    if not low:
        out["rule_determination"] = "No"
        return out

    bed_hits = find_terms(low, BED_MATCHERS)
    out["bed_elements"] = sorted(bed_hits)
    out["bed_confined"] = int(len(bed_hits) == len(BED_MATCHERS))
    out["bed_partial"] = int(0 < len(bed_hits) < len(BED_MATCHERS))
    if out["bed_confined"]:
        out["mobility_score"] += 2.0
    elif out["bed_partial"]:
        out["mobility_score"] += 0.5 * len(bed_hits)

    concept_hits = find_terms(low, CONCEPT_MATCHERS)
    out["concept_hits"] = concept_hits
    for name, evidence in concept_hits.items():
        c = CONCEPTS[name]
        if c["axis"] == "mobility":
            out["mobility_score"] += c["weight"]
        else:
            out["monitoring_score"] += c["weight"]

    cited = [n for n in concept_hits if CONCEPTS[n]["status"] == "cited"]
    out["named_score"] = 1.0 if (cited or out["bed_confined"]) else 0.0
    out["total_score"] = out["mobility_score"] + out["monitoring_score"] + out["named_score"]

    qualifying_present = bool(cited) or out["bed_confined"]
    raw_exclusions = find_terms(low, EXCLUSION_MATCHERS)
    for eid, evidence in raw_exclusions.items():
        if EXCLUSIONS[eid]["sole_reason_only"] and qualifying_present:
            continue
        out["exclusion_hits"][eid] = evidence

    if not concept_hits and not bed_hits and VAGUE_MATCHER.search(low):
        out["vague_only"] = 1

    if out["exclusion_hits"] and not qualifying_present:
        out["rule_determination"] = "No"
    elif out["total_score"] == 0.0:
        out["rule_determination"] = "No"
    elif out["total_score"] >= SCORE_NECESSARY_THRESHOLD and qualifying_present \
            and not out["exclusion_hits"]:
        out["rule_determination"] = "Yes"
    else:
        out["rule_determination"] = "Needs review"
    return out


for probe in [
    "Pt unable to get up from bed without assistance, unable to ambulate, unable to sit safely in a chair. Continuous cardiac monitoring required en route.",
    "Patient is bed confined. High fall risk.",
    "Transport to dialysis. Patient on oxygen 2L nasal cannula.",
    "Pt can sit in a wheelchair, ambulates independently. Needs transport per MD order.",
    "Combative, psychiatric hold, requires 1:1 observation and chemical restraint en route.",
]:
    r = score_order(probe)
    print(f"{r['rule_determination']:13s} total={r['total_score']:.1f} "
          f"mob={r['mobility_score']:.1f} mon={r['monitoring_score']:.1f} "
          f"bed={r['bed_elements']} excl={list(r['exclusion_hits'])}")

## 5. Extraction prompt

The prompt is compiled from the reference file, so the facility training vocabulary, the
bed confinement conjunction, and the non-covered situations all reach the model without a
separate prompt edit. The model extracts facts, quotes the text for each one, and writes a
short rationale in its own words. It does not assign the final label.

In [ ]:
def compile_prompt(kb):
    concept_keys = [c["concept"] for c in kb["concepts"]]
    exclusion_keys = [e["exclusion_id"] for e in kb["exclusions"]]
    bed_keys = [e["element"] for e in kb["bed_confinement"]["elements"]]

    concept_lines = []
    for c in sorted(kb["concepts"], key=lambda x: (x["axis"], x["concept"])):
        concept_lines.append(
            f'  "{c["concept"]}"  ({c["axis"]} axis, {c["status"]}) '
            f'- phrasing seen in practice: {", ".join(c["terms"][:8])}')

    bed_lines = [f'  "{e["element"]}" - {e["statement"]}'
                 for e in kb["bed_confinement"]["elements"]]

    exclusion_lines = [f'  "{e["exclusion_id"]}" - {e["statement"]}' for e in kb["exclusions"]]

    rule_lines = [f'  - {r["statement"]}' for r in kb["policy_rules"]]

    schema_concepts = ",\n".join(f'      "{k}": boolean' for k in concept_keys)
    schema_bed = ",\n".join(f'      "{k}": boolean' for k in bed_keys)
    schema_excl = ",\n".join(f'      "{k}": boolean' for k in exclusion_keys)

    prompt = f"""
You are a clinical documentation extraction assistant reviewing physician certification and
order text for non-emergent ground ambulance transport. Extract only what the text states.
Do not infer, do not assume, and handle negation correctly.

The governing question is: {kb["determination_question"]}
{kb["evaluation_principle"]}

BED CONFINEMENT. A patient is bed confined only when ALL THREE of the following are
documented together:
{chr(10).join(bed_lines)}
{kb["bed_confinement"]["qualifier"]}

CONDITIONS AND CARE NEEDS TO EXTRACT:
{chr(10).join(concept_lines)}

NON-COVERED SITUATIONS TO FLAG WHEN THE TEXT DESCRIBES THEM:
{chr(10).join(exclusion_lines)}

RULES THAT GOVERN EXTRACTION:
{chr(10).join(rule_lines)}

These phrases are non-specific and never on their own establish anything:
{", ".join(kb["non_specific_phrases"])}

OUTPUT valid JSON only, no prose and no code fences, all keys present, booleans never null:
{{
  "bed_confinement": {{
{schema_bed}
  }},
  "concepts": {{
{schema_concepts}
  }},
  "exclusions": {{
{schema_excl}
  }},
  "other_means_addressed": boolean,
  "non_specific_only": boolean,
  "rationale": string,
  "evidence": [{{"field": string, "value": boolean, "quote": string}}]
}}

Set a key true only when the text supports it and you can quote the text verbatim. Every true
value must appear in the evidence array with a quote copied exactly from the submitted text.
Set "other_means_addressed" true only when the text explains why another means of transport
would be unsafe. Keep "rationale" to two sentences describing what the documentation does and
does not establish. Do not state a coverage decision in the rationale.

Now extract from the following Clinical Data."""
    return concept_keys, bed_keys, exclusion_keys, prompt


CONCEPT_KEYS, BED_KEYS, EXCLUSION_KEYS, MED_NEC_PROMPT = compile_prompt(KB)
print(MED_NEC_PROMPT[:1800])
print("...")
print("prompt characters:", len(MED_NEC_PROMPT),
      "| concepts:", len(CONCEPT_KEYS), "| exclusions:", len(EXCLUSION_KEYS))

## 6. Model client

In [ ]:
from openai import OpenAI

DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN", "")
)
client = OpenAI(api_key=DATABRICKS_TOKEN or "token-not-set", base_url=WORKSPACE_BASE_URL)


def llm_call(system_prompt, user_prompt, max_tokens=LLM_MAX_TOKENS):
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
        temperature=LLM_TEMPERATURE, max_tokens=max_tokens)
    content = resp.choices[0].message.content
    if isinstance(content, list):
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                return item.get("text", "")
        return json.dumps(content)
    return content


LLM_AVAILABLE = bool(DATABRICKS_TOKEN)
print("token present:", LLM_AVAILABLE, "| model:", LLM_MODEL)

## 7. Validator

A response that does not parse, or that asserts something without quoting the text for it,
counts as a failed extraction rather than a silent default.

In [ ]:
def valid_json(raw):
    try:
        obj = json.loads(raw)
    except Exception:
        return None
    for section in ("bed_confinement", "concepts", "exclusions"):
        if section not in obj or not isinstance(obj[section], dict):
            return None

    quotes = {}
    for e in obj.get("evidence", []) or []:
        if isinstance(e, dict) and e.get("value"):
            quotes[e.get("field")] = str(e.get("quote", "") or "")

    def gate(keys, section):
        return {k: {"documented": bool(obj[section].get(k, False)) and bool(quotes.get(k, "")),
                    "evidence": quotes.get(k, "")} for k in keys}

    return {
        "bed_confinement": gate(BED_KEYS, "bed_confinement"),
        "concepts": gate(CONCEPT_KEYS, "concepts"),
        "exclusions": gate(EXCLUSION_KEYS, "exclusions"),
        "other_means_addressed": bool(obj.get("other_means_addressed", False)),
        "non_specific_only": bool(obj.get("non_specific_only", False)),
        "rationale": str(obj.get("rationale", "") or "")[:600],
    }

## 8. Determination and reasons

The same thresholds and the same exclusion logic as the rule scorer, applied to what the
model extracted. Reasons are assembled from the knowledge file, so every sentence written into
the output traces back to a cited rule or a named concept.

In [ ]:
def judge(ext):
    bed_docs = [k for k, v in ext["bed_confinement"].items() if v["documented"]]
    bed_confined = len(bed_docs) == len(BED_KEYS)
    bed_partial = 0 < len(bed_docs) < len(BED_KEYS)

    mobility = 2.0 if bed_confined else (0.5 * len(bed_docs) if bed_partial else 0.0)
    monitoring = 0.0
    concept_docs = [k for k, v in ext["concepts"].items() if v["documented"]]
    for name in concept_docs:
        c = CONCEPTS[name]
        if c["axis"] == "mobility":
            mobility += c["weight"]
        else:
            monitoring += c["weight"]

    cited = [n for n in concept_docs if CONCEPTS[n]["status"] == "cited"]
    qualifying = bool(cited) or bed_confined
    named = 1.0 if qualifying else 0.0
    total = mobility + monitoring + named

    excl_docs = []
    for eid, v in ext["exclusions"].items():
        if not v["documented"]:
            continue
        if EXCLUSIONS[eid]["sole_reason_only"] and qualifying:
            continue
        excl_docs.append(eid)

    if excl_docs and not qualifying:
        determination = "No"
    elif total == 0.0:
        determination = "No"
    elif total >= SCORE_NECESSARY_THRESHOLD and qualifying and not excl_docs:
        determination = "Yes"
    else:
        determination = "Needs review"

    reasons, supporting, against = [], [], []

    if bed_confined:
        supporting.append("Bed confinement is documented with all three required elements: "
                          "unable to get up from bed without assistance, unable to ambulate "
                          "without assistance, and unable to sit safely in a chair.")
        if not ext["other_means_addressed"]:
            against.append("Bed confinement by itself does not represent medical necessity. "
                           "The documentation does not state why another means of transport "
                           "would endanger the patient.")
    elif bed_partial:
        missing = [BED_ELEMENTS[k]["statement"] for k in BED_KEYS if k not in bed_docs]
        against.append("Bed confinement is claimed but incomplete. Missing: "
                       + "; ".join(missing) + ".")

    for name in concept_docs:
        c = CONCEPTS[name]
        label = name.replace("_", " ")
        quote = ext["concepts"][name]["evidence"]
        if c["status"] == "cited":
            supporting.append(f"{label.capitalize()} is documented ({c['axis']} axis, "
                              f"{c['source_id']}): \"{quote}\".")
        else:
            against.append(f"{label.capitalize()} is documented but is inferred from guidance "
                           f"rather than named in it, so it carries reduced weight pending SME "
                           f"validation: \"{quote}\".")

    for eid in excl_docs:
        against.append(EXCLUSIONS[eid]["statement"] + " Documented as: "
                       + f"\"{ext['exclusions'][eid]['evidence']}\".")

    if ext["other_means_addressed"]:
        supporting.append("The documentation addresses why other means of transport would "
                          "endanger the patient's health or safety.")
    else:
        against.append("The documentation does not answer the governing question of whether "
                       "the patient could be transported by any other means without "
                       "endangering their health or safety.")

    if ext["non_specific_only"]:
        against.append("The justification consists only of non-specific phrases that do not "
                       "describe the patient's condition.")

    if mobility <= 0:
        against.append("No mobility limitation is documented.")
    if monitoring <= 0:
        against.append("No monitoring or care need during transport is documented.")

    reasons = supporting + against

    return {
        "med_nec": determination,
        "mobility_score": round(mobility, 2),
        "monitoring_score": round(monitoring, 2),
        "named_score": named,
        "total_score": round(total, 2),
        "bed_elements_documented": "|".join(sorted(bed_docs)),
        "bed_confined": int(bed_confined),
        "concepts_documented": "|".join(sorted(concept_docs)),
        "exclusions_triggered": "|".join(sorted(excl_docs)),
        "other_means_addressed": int(ext["other_means_addressed"]),
        "supporting_reasons": " ".join(supporting) if supporting else "None.",
        "reasons_against": " ".join(against) if against else "None.",
        "reasons": " ".join(reasons),
        "llm_rationale": ext["rationale"],
        "evidence_quotes": "; ".join(
            f"{k}: {v['evidence']}" for k, v in
            list(ext["concepts"].items()) + list(ext["bed_confinement"].items())
            + list(ext["exclusions"].items()) if v["documented"] and v["evidence"]),
    }

## 9. Transport.net orders

Column names are resolved rather than assumed. Rideshare and emergent transports are excluded
so they do not inflate the at-risk denominator. Sampling is done Spark side with a fixed seed
so the same orders come back on a re-run.

In [ ]:
def find_col(df, candidates, contains=None):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    if contains:
        for lc, orig in cols.items():
            if all(tok in lc for tok in contains):
                return orig
    return None


src = spark.table(SOURCE_TABLE)
print("source columns:", len(src.columns))

ORDER_ID = find_col(src, ["order_id", "trip_id", "transport_id"], ["order", "id"])
NOTES = find_col(src, ["medical_necessity_notes", "clinical_notes", "reason_for_transport",
                       "justification", "notes"], ["reason"])
FACILITY = find_col(src, ["facility_name", "origin_facility", "pickup_facility"], ["facility"])
SERVICE = find_col(src, ["service_level", "level_of_service", "service_code"], ["service"])
CREATED = find_col(src, ["created_date", "order_date", "transport_date"], ["date"])
PAYER = find_col(src, ["payer", "payer_name", "insurance"], ["payer"])

resolved = pd.DataFrame([
    {"role": "order_id", "column": ORDER_ID},
    {"role": "notes", "column": NOTES},
    {"role": "facility", "column": FACILITY},
    {"role": "service_level", "column": SERVICE},
    {"role": "created", "column": CREATED},
    {"role": "payer", "column": PAYER},
])
print(resolved.to_string(index=False))

missing = resolved[resolved["column"].isna()]["role"].tolist()
if missing:
    print("UNRESOLVED, set these by hand before continuing:", missing)

keep = [c for c in [ORDER_ID, CREATED, FACILITY, SERVICE, PAYER, NOTES] if c]
scoped = src.select(*keep)
if SERVICE:
    for code in EXCLUDED_SERVICE_CODES:
        scoped = scoped.filter(~SF.upper(SF.col(SERVICE)).contains(code))
if NOTES:
    scoped = scoped.filter(SF.length(SF.trim(SF.col(NOTES))) >= MIN_TEXT_CHARS)

total_scoped = scoped.count()
sample = scoped.sample(withReplacement=False,
                       fraction=min(1.0, (ORDER_LIMIT * 3.0) / max(total_scoped, 1)),
                       seed=SAMPLE_SEED).limit(ORDER_LIMIT)

orders = sample.toPandas()
orders = orders.rename(columns={
    ORDER_ID: "order_id", NOTES: "notes", FACILITY: "facility",
    SERVICE: "service_level", CREATED: "created", PAYER: "payer"})
for col in ["order_id", "notes", "facility", "service_level", "created", "payer"]:
    if col not in orders.columns:
        orders[col] = ""

print()
print("orders in scope after exclusions:", f"{total_scoped:,}")
print("orders selected for review:", len(orders))
print()
print(orders[["order_id", "facility", "service_level"]].head(5).to_string(index=False))

## 10. Review

One call per order. A response that fails the validator is recorded as a failed extraction and
falls through to the rule scorer, which is flagged in the `review_path` column so a reader can
tell which determinations the model produced.

In [ ]:
rows = []
for r in orders.to_dict("records"):
    user_prompt = (f"order_id: {r['order_id']}\n"
                   f"facility: {r['facility']}\n"
                   f"service_level: {r['service_level']}\n\n"
                   f"Clinical Data:\n{r['notes']}")
    obj = None
    if LLM_AVAILABLE:
        try:
            obj = valid_json(llm_call(MED_NEC_PROMPT, user_prompt))
        except Exception:
            obj = None
    rows.append({"order_id": r["order_id"], "ok": obj is not None, "obj": obj})

ext_status = pd.DataFrame(rows)
print(f"Valid extractions: {ext_status['ok'].mean() * 100:.1f}%  "
      f"({int(ext_status['ok'].sum())} of {len(ext_status)})")

records = []
for r, e in zip(orders.to_dict("records"), rows):
    rule = score_order(r["notes"])
    if e["ok"]:
        verdict = judge(e["obj"])
        path = "llm"
    else:
        verdict = {
            "med_nec": rule["rule_determination"],
            "mobility_score": round(rule["mobility_score"], 2),
            "monitoring_score": round(rule["monitoring_score"], 2),
            "named_score": rule["named_score"],
            "total_score": round(rule["total_score"], 2),
            "bed_elements_documented": "|".join(rule["bed_elements"]),
            "bed_confined": rule["bed_confined"],
            "concepts_documented": "|".join(sorted(rule["concept_hits"])),
            "exclusions_triggered": "|".join(sorted(rule["exclusion_hits"])),
            "other_means_addressed": int("contraindicated_other_means" in rule["concept_hits"]),
            "supporting_reasons": "Rule-based fallback, model extraction unavailable.",
            "reasons_against": "Rule-based fallback, model extraction unavailable.",
            "reasons": "Determined by keyword rules because the model extraction failed "
                       "validation for this order.",
            "llm_rationale": "",
            "evidence_quotes": "; ".join(f"{k}: {v}" for k, v in rule["concept_hits"].items()),
        }
        path = "rule_fallback"

    rec = {
        "order_id": r["order_id"],
        "created": r["created"],
        "facility": r["facility"],
        "service_level": r["service_level"],
        "payer": r["payer"],
        "notes": r["notes"],
        "review_path": path,
        "rule_determination": rule["rule_determination"],
    }
    rec.update(verdict)
    rec["agreement"] = int(rec["med_nec"] == rule["rule_determination"])
    records.append(rec)

reviewed = pd.DataFrame(records)

COLUMN_ORDER = ["order_id", "created", "facility", "service_level", "payer", "notes",
                "med_nec", "llm_rationale", "supporting_reasons", "reasons_against",
                "evidence_quotes", "bed_confined", "bed_elements_documented",
                "concepts_documented", "exclusions_triggered", "other_means_addressed",
                "mobility_score", "monitoring_score", "named_score", "total_score",
                "review_path", "rule_determination", "agreement"]
reviewed = reviewed[[c for c in COLUMN_ORDER if c in reviewed.columns]]

print()
print(reviewed["med_nec"].value_counts().to_string())
print()
print("agreement with rule scorer:", f"{reviewed['agreement'].mean() * 100:.1f}%")
print()
print(reviewed[["order_id", "med_nec", "exclusions_triggered", "total_score"]]
      .head(10).to_string(index=False))

## 11. Why each order landed where it did

In [ ]:
for r in reviewed.head(5).to_dict("records"):
    print("=" * 100)
    print("order:", r["order_id"], "|", r["facility"], "|", r["med_nec"])
    print("-" * 100)
    print("notes:", (r["notes"] or "")[:400])
    print()
    print("model rationale:", r["llm_rationale"])
    print()
    print("supporting:", r["supporting_reasons"])
    print()
    print("against:", r["reasons_against"])
    print()

by_exclusion = (reviewed[reviewed["exclusions_triggered"] != ""]
                .assign(exclusion=lambda d: d["exclusions_triggered"].str.split("|"))
                .explode("exclusion")
                .groupby("exclusion").size().rename("orders")
                .sort_values(ascending=False).reset_index())
print("=" * 100)
print("Exclusions triggered across the reviewed orders")
print(by_exclusion.to_string(index=False) if len(by_exclusion) else "none")

## 12. Output

One Orders tab with autofilter carries the deliverable: the order, its notes, the
determination, and the reasons. The knowledge tabs are included so a reader can check any
reason back to its source without opening the JSON.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def sanitize(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, (int, float)):
        return v
    return CONTROL_CHARS.sub(" ", str(v))


TABS = [
    ("Orders", reviewed),
    ("Concepts", concept_frame),
    ("Exclusions", exclusion_frame),
    ("Bed Confinement", pd.DataFrame(
        [{"element": e["element"], "statement": e["statement"]} for e in BED["elements"]]
        + [{"element": "qualifier", "statement": BED["qualifier"]}])),
    ("Policy Rules", rule_frame),
    ("Sources", source_frame),
]

WIDTHS = {"order_id": 18, "created": 14, "facility": 26, "service_level": 12, "payer": 16,
          "notes": 70, "med_nec": 13, "llm_rationale": 60, "supporting_reasons": 70,
          "reasons_against": 70, "evidence_quotes": 50, "concepts_documented": 30,
          "exclusions_triggered": 26, "bed_elements_documented": 30, "statement": 90,
          "element": 30, "exclusion_id": 26, "concept": 26, "cite": 22, "provenance": 55}

wb = Workbook()
wb.remove(wb.active)
for name, frame in TABS:
    ws = wb.create_sheet(name[:31])
    ws.append([sanitize(c) for c in frame.columns])
    for cell in ws[1]:
        cell.font = Font(name="Calibri", bold=True)
    for rec in frame.itertuples(index=False):
        ws.append([sanitize(v) for v in rec])
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for i, col in enumerate(frame.columns, start=1):
        ws.column_dimensions[get_column_letter(i)].width = WIDTHS.get(str(col), 16)

xlsx_name = f"med_nec_order_review_{RUN_TS}.xlsx"
local_xlsx = os.path.join(LOCAL_DIR, xlsx_name)
wb.save(local_xlsx)

try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    shutil.copyfile(local_xlsx, os.path.join(OUTPUT_DIR, xlsx_name))
    print("wrote", os.path.join(OUTPUT_DIR, xlsx_name))
except Exception as e:
    print("workspace copy skipped:", e)
    print("local copy at", local_xlsx)

print()
print("orders reviewed:", len(reviewed))
print(reviewed["med_nec"].value_counts().to_string())

## 13. Maintaining the knowledge file

`med_nec_knowledge.json` is the only place criteria live. Nothing in this notebook hard-codes
a term, a weight, an exclusion, or a rule statement.

To add a concept, append to `concepts` with an axis, a weight, a status of `cited` or
`inferred`, a source and cite, and the phrasing seen in practice. To add a non-covered
situation, append to `exclusions` and set `sole_reason_only` to true when it should only rule
an order out in the absence of anything qualifying.

Open items for Jen and Michelle:

- Bariatric, wound and ostomy, and isolation remain inferred. They are not named in the CMS
  text or in the 2026-08-13 facility training and carry reduced weight until confirmed.
- Behavioral is no longer inferred. The training names psychiatric emergencies directly, so
  the concept is now `psychiatric_emergency` with cited status and full weight.
- `DAVE_JEN` is still marked pending. The summary of the discussion with Jen has not been
  ingested, and its rules currently duplicate what the training already establishes.
- Oxygen appears on both sides. Oxygen regulation is a qualifying monitoring need; a patient
  on oxygen with no regulation need is a named non-covered situation. Confirm the boundary
  reads correctly against real orders.